# 03_Feature_Extraction.ipynb

Reads Sentinel/Landsat GeoTIFFs from Google Drive, computes indices, and writes features.csv.

In [ ]:

!pip -q install rasterio numpy pandas tqdm matplotlib

from google.colab import drive
drive.mount('/content/drive')

import os, glob
import rasterio
import numpy as np
import pandas as pd
from tqdm import tqdm

ROOT='/content/drive/MyDrive/FloodProject'
SENT=os.path.join(ROOT,'Sentinel')
LAND=os.path.join(ROOT,'Landsat')
OUT=os.path.join(ROOT,'processed_images')
os.makedirs(OUT,exist_ok=True)

def idx(a,b):
    a=a.astype('float32')
    b=b.astype('float32')
    return np.where((a+b)==0,np.nan,(a-b)/(a+b))

def save(arr,profile,path):
    p=profile.copy()
    p.update(dtype='float32',count=1)
    with rasterio.open(path,'w',**p) as dst:
        dst.write(arr.astype('float32'),1)

rows=[]
files=glob.glob(os.path.join(SENT,'*.tif'))+glob.glob(os.path.join(LAND,'*.tif'))

for f in tqdm(files):
    with rasterio.open(f) as src:
        img=src.read()
        prof=src.profile
    sat='Sentinel-2' if 'Sentinel' in f else 'Landsat'
    nir,red,green,blue=img[0],img[1],img[2],img[3]
    ndvi=idx(nir,red)
    ndwi=idx(green,nir)
    ndmi=idx(nir,red)
    mndwi=idx(green,blue)
    ndbi=idx(red,nir)
    name=os.path.splitext(os.path.basename(f))[0]
    save(ndvi,prof,os.path.join(OUT,name+'_NDVI.tif'))
    save(ndwi,prof,os.path.join(OUT,name+'_NDWI.tif'))
    rows.append({
        'event_id':name,
        'satellite':sat,
        'NDVI_mean':float(np.nanmean(ndvi)),
        'NDWI_mean':float(np.nanmean(ndwi)),
        'NDMI_mean':float(np.nanmean(ndmi)),
        'MNDWI_mean':float(np.nanmean(mndwi)),
        'NDBI_mean':float(np.nanmean(ndbi))
    })

pd.DataFrame(rows).to_csv(os.path.join(ROOT,'features.csv'),index=False)
print('Done')
